# Lab 3.1 - Amazon SageMaker: Creating and importing data

**Educate edition.** This notebook replaces the Vocareum-provisioned
`en_us/3_1-machinelearning.ipynb`. It runs on any SageMaker notebook
instance (or any Jupyter kernel with internet access).

## Objectives
* Run code and Markdown cells in a Jupyter notebook
* Download a dataset from an external source
* Load an ARFF file into a pandas DataFrame
* Persist the data so labs 3.2-3.7 can reuse it

**Cost note:** this notebook uses *no* AWS services beyond the notebook
instance itself. Everything here is free once the instance is running.

## Step 1 - Install and import dependencies

`scipy` provides the ARFF reader. On most SageMaker `conda_python3`
kernels it is already present, so the install is a no-op.

In [ ]:
!pip install --quiet scipy pandas matplotlib seaborn

import warnings, requests, zipfile, io, os
warnings.simplefilter('ignore')

import pandas as pd
from scipy.io import arff

print('pandas', pd.__version__)

## Step 2 - Download and extract the dataset

The dataset is the **Vertebral Column** dataset from the UC Irvine
Machine Learning Repository: 310 orthopaedic patients described by six
biomechanical features, labelled `Normal` or `Abnormal`.

The file is a small ZIP, so we stream it into memory and extract it
directly rather than saving the archive to disk.

Two mirrors are tried; the second is UCI's newer static path.

In [ ]:
URLS = [
    'http://archive.ics.uci.edu/ml/machine-learning-databases/00212/vertebral_column_data.zip',
    'https://archive.ics.uci.edu/static/public/212/vertebral+column.zip',
]

extracted = False
for url in URLS:
    try:
        r = requests.get(url, stream=True, timeout=60)
        r.raise_for_status()
        zipfile.ZipFile(io.BytesIO(r.content)).extractall()
        print('Downloaded and extracted from:', url)
        extracted = True
        break
    except Exception as e:
        print('Failed:', url, '->', type(e).__name__, e)

if not extracted:
    print('All mirrors failed. Run the offline fallback cell below.')

### Confirm the extracted files

You should now see four files in the working directory:

| File | Contents |
|---|---|
| `column_2C_weka.arff` | 2 classes (Normal / Abnormal), ARFF format |
| `column_2C.dat` | same data, space-delimited, no header |
| `column_3C_weka.arff` | 3 classes (Normal / Disk Hernia / Spondylolisthesis) |
| `column_3C.dat` | same, space-delimited |

The labs use the **two-class** file.

In [ ]:
for f in sorted(os.listdir('.')):
    if f.startswith('column'):
        print(f, os.path.getsize(f), 'bytes')

### Offline fallback (only run if the download failed)

Use this only to keep going if your account has no
outbound internet access. It generates synthetic data with the same
shape and roughly the same statistics.

In [ ]:
import numpy as np

if not extracted:
    rng = np.random.default_rng(0)
    cols = ['pelvic_incidence','pelvic_tilt','lumbar_lordosis_angle',
            'sacral_slope','pelvic_radius','degree_spondylolisthesis']
    n_ab, n_no = 210, 100
    ab = rng.normal([64,19,55,45,115,37], [12,10,18,13,13,40], size=(n_ab,6))
    no = rng.normal([51,13,43,38,124,2],  [10,7,12,10,9,6],    size=(n_no,6))
    df_fb = pd.DataFrame(np.vstack([ab,no]), columns=cols)
    df_fb['class'] = ['Abnormal']*n_ab + ['Normal']*n_no
    df_fb = df_fb.sample(frac=1, random_state=0).reset_index(drop=True)
    df_fb.to_csv('vertebral_column.csv', index=False)
    print('Offline fallback written to vertebral_column.csv')
    print('NOTE: synthetic data, for demo continuity only.')
else:
    print('Download succeeded - skip this cell.')

## Step 3 - Load the ARFF file into a pandas DataFrame

`arff.loadarff` returns a tuple of `(data, metadata)`. The data is a
NumPy structured array, which pandas consumes directly.

One gotcha: the `class` column arrives as **bytes** (`b'Abnormal'`),
not `str`, because ARFF nominal values are byte strings. We decode it.

In [ ]:
if extracted:
    data = arff.loadarff('column_2C_weka.arff')
    df = pd.DataFrame(data[0])
    df['class'] = df['class'].str.decode('utf-8')
else:
    df = pd.read_csv('vertebral_column.csv')

df.head()

## Step 4 - Sanity-check what we loaded

In [ ]:
print('Shape:', df.shape)
print('Columns:', list(df.columns))
print()
print('Class balance:')
print(df['class'].value_counts())
print()
print('Missing values per column:')
print(df.isnull().sum())

## Step 5 - Persist the data for the later labs

Labs 3.2 and 3.4-3.7 all start from this same DataFrame. Saving it to
CSV means each lab reloads it in one line, and it survives a kernel
restart.

The file lives on the notebook instance's EBS volume, so it persists
across **stop/start** of the same instance, but is lost if you
**delete** the instance.

In [ ]:
df.to_csv('vertebral_column.csv', index=False)
print('Saved vertebral_column.csv ->', len(df), 'rows')

## Conclusion

You have:
* Run code and Markdown cells in JupyterLab
* Downloaded and extracted a dataset from an external source
* Loaded an ARFF file into a pandas DataFrame
* Saved the data for reuse in labs 3.2-3.7

**Before you walk away:** go back to the SageMaker console and **Stop**
the notebook instance. A stopped instance costs nothing per hour.

Next: `3_2-machinelearning.ipynb`